In [0]:
%run ./env

In [0]:
%run ./python_libraries

# phase2_migration_schema — à exécuter UNE SEULE FOIS par environnement

La phase 2 ajoute des colonnes à deux tables existantes du modèle :

| Table | Colonnes ajoutées |
|---|---|
| `dim_batches_specifications` | `id_good_specy`, `id_good_variety`, `id_parameter_production_type` |
| `fact_batch_note` | `key_location`, `key_impact`, `key_event`, `key_detail` |

## Pourquoi ce notebook est nécessaire

`handle_table_update` **ne fait aucune évolution de schéma**, dans aucun de ses
deux modes :

- en mode `update`, le `whenNotMatchedInsert` référence `source.<colonne>` pour
  toutes les colonnes du DataFrame ; une colonne absente de la table cible fait
  échouer le MERGE ;
- en mode `full`, le `write.mode("append")` après le `delete()` échoue tout
  autant sur une différence de schéma.

Autrement dit, **relancer en mode `full` ne résout pas le problème** : il faut
ajouter les colonnes explicitement, une fois, avant le premier run du pipeline
modifié.

`ALTER TABLE ... ADD COLUMNS` est une opération de métadonnées sur Delta : elle
ne réécrit pas les données et les lignes existantes prennent `NULL` sur les
nouvelles colonnes. Ces `NULL` seront remplis au premier passage du pipeline.

In [0]:
target_dim_batches_specifications = f"{current_catalog}.{current_schema}.dim_batches_specifications"
target_fact_batch_note = f"{current_catalog}.{current_schema}.fact_batch_note"

colonnes_a_ajouter = {
    target_dim_batches_specifications: [
        ("id_good_specy", "INT"),
        ("id_good_variety", "INT"),
        ("id_parameter_production_type", "INT"),
    ],
    target_fact_batch_note: [
        ("key_location", "STRING"),
        ("key_impact", "STRING"),
        ("key_event", "STRING"),
        ("key_detail", "STRING"),
    ],
}

In [0]:
for table_name, colonnes in colonnes_a_ajouter.items():
    if not spark.catalog.tableExists(table_name):
        print(f"{table_name} : table absente, rien à faire (elle sera créée au premier run).")
        continue

    existantes = set(spark.table(table_name).columns)
    manquantes = [(nom, typ) for nom, typ in colonnes if nom not in existantes]

    if not manquantes:
        print(f"{table_name} : déjà à jour.")
        continue

    ddl = ", ".join(f"{nom} {typ}" for nom, typ in manquantes)
    spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({ddl})")
    print(f"{table_name} : colonnes ajoutées -> {ddl}")

## Contrôle

Les colonnes doivent apparaître, à `NULL` sur les lignes existantes tant que le
pipeline n'a pas été relancé.

In [0]:
for table_name in colonnes_a_ajouter:
    if spark.catalog.tableExists(table_name):
        print(f"--- {table_name}")
        display(spark.table(table_name).limit(5))